# Lab 6 - Deep Learning for NLP

In this lab a neural network classifies Yelp restaurant reviews as negative (0) or positive (1). The lab used TF-IDF vectors. In the tasks here I use Word2Vec vectors instead and build a different network.

Libraries used: pandas, scikit-learn, nltk, gensim, PyTorch.

Install the libraries first if they are missing: `pip install torch pandas scikit-learn nltk gensim`

In [1]:
import warnings
warnings.filterwarnings("ignore")

import re

import nltk
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

_ = nltk.download("punkt", quiet=True)
_ = nltk.download("punkt_tab", quiet=True)

torch.manual_seed(42)
np.random.seed(42)

## Load, preprocess and split the Yelp data

This part is the same as in the lab. One difference: in the lab code the line that applies `preprocess_text` is indented inside the function after the `return`, so it never runs. Here I apply it to the dataframe.

In [2]:
df = pd.read_csv("dataset/07-yelp-dataset.txt", sep="\t", names=["review", "label"])
print("Number of reviews:", len(df))
print(df["label"].value_counts())
df.head()

Number of reviews: 1000
label
1    500
0    500
Name: count, dtype: int64


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
def preprocess_text(text):
    text = text.lower()                          # lowercase
    text = re.sub(r"[^a-zA-Z\s]", "", text)      # remove punctuation and special characters
    text = re.sub(r"\s+", " ", text).strip()      # remove extra spaces
    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


In [4]:
X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## Task 1: Represent reviews using Word2Vec

1. Tokenize the training and testing reviews.
2. Train Word2Vec on the **training** reviews only (`vector_size=200`, `window=5`, `min_count=1`, `sg=1`).
3. Convert each review to one vector by averaging the vectors of its words. Words that are not in the Word2Vec vocabulary (some test words) are skipped.
4. Convert the vectors to PyTorch tensors.

In [5]:
train_tokens = [word_tokenize(review) for review in X_train]
test_tokens = [word_tokenize(review) for review in X_test]

print(train_tokens[0])

['it', 'is', 'perfect', 'for', 'a', 'sitdown', 'family', 'meal', 'or', 'get', 'together', 'with', 'a', 'few', 'friends']


In [6]:
w2v = Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1, seed=42, workers=1)
print("Word2Vec vocabulary size:", len(w2v.wv))

Word2Vec vocabulary size: 1786


In [7]:
def review_to_vector(tokens):
    # average of the word vectors of the review
    vectors = [w2v.wv[w] for w in tokens if w in w2v.wv]
    if len(vectors) == 0:
        return np.zeros(w2v.vector_size)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([review_to_vector(t) for t in train_tokens])
X_test_w2v = np.array([review_to_vector(t) for t in test_tokens])

print("Train vectors:", X_train_w2v.shape)
print("Test vectors:", X_test_w2v.shape)

Train vectors: (800, 200)
Test vectors: (200, 200)


In [8]:
X_train_tensor = torch.FloatTensor(X_train_w2v)
X_test_tensor = torch.FloatTensor(X_test_w2v)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

print(X_train_tensor.shape, y_train_tensor.shape)
print(X_test_tensor.shape, y_test_tensor.shape)

torch.Size([800, 200]) torch.Size([800, 1])
torch.Size([200, 200]) torch.Size([200, 1])


## Task 2: Build a different neural network

Architecture: input (200) -> 128 -> 64 -> 32 -> 1 output neuron, with ReLU after each hidden layer. There is no Sigmoid in the model because `BCEWithLogitsLoss` already applies it.

Training: `BCEWithLogitsLoss`, Adam with learning rate 0.001, 15 epochs.

In [9]:
model = nn.Sequential(
    nn.Linear(200, 128),   # input layer -> hidden layer 1
    nn.ReLU(),
    nn.Linear(128, 64),    # hidden layer 2
    nn.ReLU(),
    nn.Linear(64, 32),     # hidden layer 3
    nn.ReLU(),
    nn.Linear(32, 1)       # output neuron
)
print(model)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [10]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)              # forward pass
    loss = loss_function(outputs, y_train_tensor)  # loss
    loss.backward()                              # backpropagation
    optimizer.step()                             # update the weights

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 1/15 - Loss: 0.6958
Epoch 2/15 - Loss: 0.6953
Epoch 3/15 - Loss: 0.6949
Epoch 4/15 - Loss: 0.6946
Epoch 5/15 - Loss: 0.6943
Epoch 6/15 - Loss: 0.6940
Epoch 7/15 - Loss: 0.6938
Epoch 8/15 - Loss: 0.6936
Epoch 9/15 - Loss: 0.6934
Epoch 10/15 - Loss: 0.6933
Epoch 11/15 - Loss: 0.6932
Epoch 12/15 - Loss: 0.6932
Epoch 13/15 - Loss: 0.6932
Epoch 14/15 - Loss: 0.6932
Epoch 15/15 - Loss: 0.6932


In [11]:
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)
print(f"Test Accuracy: {accuracy:.3f}")
print()
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Test Accuracy: 0.495

              precision    recall  f1-score   support

    Negative       0.50      0.99      0.66       100
    Positive       0.00      0.00      0.00       100

    accuracy                           0.49       200
   macro avg       0.25      0.49      0.33       200
weighted avg       0.25      0.49      0.33       200

Confusion Matrix:
[[ 99   1]
 [100   0]]


## Observations

- The Word2Vec model learned a vocabulary of 1786 words from the 800 training reviews, and each review became one 200-dimensional vector (average of its word vectors).
- The loss only went from 0.6958 to 0.6932 over the 15 epochs. 0.6932 is about ln(2), which is what a network gets when it just guesses.
- The test accuracy is 0.495, about chance for two balanced classes. The network predicted almost everything as negative (recall 0.99 for negative, 0.00 for positive).
- Reason: 800 reviews is a very small corpus for Word2Vec, so the averaged vectors are close to each other, and 15 full-batch steps with lr 0.001 is not enough training. More epochs, a higher learning rate or pretrained vectors should help.
